# Best Practices for Preprocessing Natural Language Data

In this notebook, we improve the quality of our Project Gutenberg word vectors by adopting best-practices for preprocessing natural language data.

**N.B.:** Some, all or none of these preprocessing steps may be helpful to a given downstream application. 

#### Load dependencies

In [1]:
# the initial block is copied from creating_word_vectors_with_word2vec.ipynb
import nltk
from nltk import word_tokenize, sent_tokenize
import gensim
from gensim.models.word2vec import Word2Vec
from sklearn.manifold import TSNE
import pandas as pd
from bokeh.io import output_notebook, output_file
from bokeh.plotting import show, figure
%matplotlib inline

In [2]:
# nltk.download('punkt')

In [3]:
# new!
import string
from nltk.corpus import stopwords
from nltk.stem.porter import *
from gensim.models.phrases import Phraser, Phrases
from keras.preprocessing.text import one_hot

Using TensorFlow backend.
/anaconda3/lib/python3.5/importlib/_bootstrap.py:222: RuntimeWarning: compiletime version 3.6 of module 'tensorflow.python.framework.fast_tensor_util' does not match runtime version 3.5
  return f(*args, **kwds)


In [4]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/garvitkhurana/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### Load data

In [5]:
nltk.download('gutenberg')

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/garvitkhurana/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [6]:
from nltk.corpus import gutenberg

In [7]:
gberg_sents = gutenberg.sents()

#### Iteratively preprocess a sentence

##### a tokenized sentence: 

In [8]:
gberg_sents[4]

['She',
 'was',
 'the',
 'youngest',
 'of',
 'the',
 'two',
 'daughters',
 'of',
 'a',
 'most',
 'affectionate',
 ',',
 'indulgent',
 'father',
 ';',
 'and',
 'had',
 ',',
 'in',
 'consequence',
 'of',
 'her',
 'sister',
 "'",
 's',
 'marriage',
 ',',
 'been',
 'mistress',
 'of',
 'his',
 'house',
 'from',
 'a',
 'very',
 'early',
 'period',
 '.']

##### to lowercase: 

In [9]:
[w.lower() for w in gberg_sents[4]]

['she',
 'was',
 'the',
 'youngest',
 'of',
 'the',
 'two',
 'daughters',
 'of',
 'a',
 'most',
 'affectionate',
 ',',
 'indulgent',
 'father',
 ';',
 'and',
 'had',
 ',',
 'in',
 'consequence',
 'of',
 'her',
 'sister',
 "'",
 's',
 'marriage',
 ',',
 'been',
 'mistress',
 'of',
 'his',
 'house',
 'from',
 'a',
 'very',
 'early',
 'period',
 '.']

##### remove stopwords and punctuation: 

In [10]:
stpwrds = stopwords.words('english') + list(string.punctuation) ## for removing punctutation

In [11]:
stpwrds

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

In [12]:
[w.lower() for w in gberg_sents[4] if w not in stpwrds]

['she',
 'youngest',
 'two',
 'daughters',
 'affectionate',
 'indulgent',
 'father',
 'consequence',
 'sister',
 'marriage',
 'mistress',
 'house',
 'early',
 'period']

##### stem words: 

In [13]:
stemmer = PorterStemmer()

In [14]:
[stemmer.stem(w.lower()) for w in gberg_sents[4] if w not in stpwrds]

['she',
 'youngest',
 'two',
 'daughter',
 'affection',
 'indulg',
 'father',
 'consequ',
 'sister',
 'marriag',
 'mistress',
 'hous',
 'earli',
 'period']

##### handle bigram collocations:

In [15]:
phrases = Phrases(gberg_sents) # train detector

In [16]:
bigram = Phraser(phrases) # create a more efficient Phraser object for transforming sentences

In [17]:
bigram.phrasegrams # output count and score of each bigram

{(b'Miss', b'Woodhouse'): (173, 294.52709701744294),
 (b"'", b'tis'): (111, 27.952638060458856),
 (b'committed', b'adultery'): (7, 279.32294489611564),
 (b'be', b'quenched'): (10, 12.197460395102247),
 (b'wherefore', b'then'): (12, 14.182341474601536),
 (b'poor', b'fellow'): (31, 72.63216713721062),
 (b'"', b'Perhaps'): (40, 10.879118421244423),
 (b'God', b'Almighty'): (15, 14.224259082492841),
 (b'Mrs', b'Smith'): (64, 111.99977591965032),
 (b'roaring', b'lion'): (8, 261.8948334274421),
 (b'In', b'short'): (23, 17.859158861411398),
 (b'-', b'edged'): (8, 19.057657935285057),
 (b'turning', b'round'): (15, 23.423618265559664),
 (b'high', b'above'): (15, 13.152940275683413),
 (b'twenty', b'years'): (69, 85.29044131115464),
 (b'better', b'than'): (170, 42.51368036077655),
 (b'little', b'boy'): (63, 26.45163569321534),
 (b'(', b'FOLIO'): (7, 78.03911918733043),
 (b'stirred', b'up'): (21, 39.15528382897054),
 (b'drink', b'tea'): (7, 32.50399453379586),
 (b'are', b'mistaken'): (17, 10.345294

In [18]:
"Jon lives in New York City".split()

['Jon', 'lives', 'in', 'New', 'York', 'City']

In [19]:
bigram["Jon lives in New York City".split()]

['Jon', 'lives', 'in', 'New_York', 'City']

#### Preprocess the corpus

In [20]:
lower_sents = []
for s in gberg_sents:
    lower_sents.append([w.lower() for w in s if w not in list(string.punctuation)])

In [21]:
lower_sents[0:5]

[['emma', 'by', 'jane', 'austen', '1816'],
 ['volume', 'i'],
 ['chapter', 'i'],
 ['emma',
  'woodhouse',
  'handsome',
  'clever',
  'and',
  'rich',
  'with',
  'a',
  'comfortable',
  'home',
  'and',
  'happy',
  'disposition',
  'seemed',
  'to',
  'unite',
  'some',
  'of',
  'the',
  'best',
  'blessings',
  'of',
  'existence',
  'and',
  'had',
  'lived',
  'nearly',
  'twenty',
  'one',
  'years',
  'in',
  'the',
  'world',
  'with',
  'very',
  'little',
  'to',
  'distress',
  'or',
  'vex',
  'her'],
 ['she',
  'was',
  'the',
  'youngest',
  'of',
  'the',
  'two',
  'daughters',
  'of',
  'a',
  'most',
  'affectionate',
  'indulgent',
  'father',
  'and',
  'had',
  'in',
  'consequence',
  'of',
  'her',
  'sister',
  's',
  'marriage',
  'been',
  'mistress',
  'of',
  'his',
  'house',
  'from',
  'a',
  'very',
  'early',
  'period']]

In [22]:
lower_bigram = Phraser(Phrases(lower_sents))

In [23]:
lower_bigram.phrasegrams # miss taylor, mr woodhouse, mr weston

{(b'thy', b'redeemer'): (7, 10.273183404222806),
 (b'loose', b'fish'): (15, 128.85949779566803),
 (b'headed', b'whale'): (6, 10.265601140181207),
 (b'cried', b'townsend'): (8, 48.225251076040166),
 (b'committed', b'adultery'): (7, 266.76984126984127),
 (b'be', b'quenched'): (10, 10.429103319888304),
 (b'poor', b'fellow'): (38, 75.64730219711522),
 (b'after', b'breakfast'): (11, 10.176820020576768),
 (b'exclaimed', b'mrs'): (11, 11.847730944338375),
 (b'cried', b'ahab'): (32, 24.39161459383189),
 (b'roaring', b'lion'): (9, 282.06713286713284),
 (b'beautiful', b'soup'): (8, 307.85312075983717),
 (b'an', b'hundred'): (186, 29.194395140414738),
 (b'turning', b'round'): (15, 21.605116375400833),
 (b'guernsey', b'man'): (11, 53.87658058771149),
 (b'suburbs', b'6'): (15, 21.77007772020725),
 (b'better', b'than'): (175, 40.46903672973475),
 (b'little', b'boy'): (67, 21.623428772358615),
 (b'stirred', b'up'): (22, 39.37877302868344),
 (b'drink', b'tea'): (7, 29.38199300699301),
 (b'tenth', b'pa

In [24]:
lower_bigram["jon lives in new york city".split()]

['jon', 'lives', 'in', 'new_york', 'city']

In [25]:
lower_bigram = Phraser(Phrases(lower_sents, min_count=32, threshold=64))
lower_bigram.phrasegrams

{(b'afar', b'off'): (52, 108.14220347465505),
 (b'burnt', b'offering'): (184, 297.524653753951),
 (b'burnt', b'offerings'): (86, 299.15702343127646),
 (b'buster', b'bear'): (142, 479.87410772225826),
 (b'captain', b'benwick'): (56, 241.49037086312987),
 (b'captain', b'wentworth'): (196, 529.8756608388247),
 (b'charles', b'hayter'): (33, 92.03437785214481),
 (b'chief', b'priests'): (65, 116.31947753846512),
 (b'colonel', b'brandon'): (132, 1313.0078125),
 (b'couldn', b't'): (89, 171.76138536935215),
 (b'cut', b'off'): (217, 129.60290535032792),
 (b'dare', b'say'): (115, 89.94000515807346),
 (b'de', b'grey'): (77, 603.2109624246722),
 (b'didn', b't'): (180, 220.51081560283686),
 (b'doesn', b't'): (53, 106.2634985949418),
 (b'don', b't'): (830, 250.30957446808512),
 (b'dr', b'bull'): (65, 680.7870294599019),
 (b'dr', b'middleton'): (40, 162.73103819257668),
 (b'drawing', b'room'): (49, 84.91494947493561),
 (b'farmer', b'brown'): (100, 386.05179596892236),
 (b'father', b'brown'): (207, 91.

In [26]:
# as in Maas et al. (2001):
# - leave in stop words ("indicative of sentiment")
# - no stemming ("model learns similar representations of words of the same stem when data suggests it")
clean_sents = []
for s in lower_sents:
    clean_sents.append(lower_bigram[s])

In [27]:
clean_sents[0:9]

[['emma', 'by', 'jane', 'austen', '1816'],
 ['volume', 'i'],
 ['chapter', 'i'],
 ['emma',
  'woodhouse',
  'handsome',
  'clever',
  'and',
  'rich',
  'with',
  'a',
  'comfortable',
  'home',
  'and',
  'happy',
  'disposition',
  'seemed',
  'to',
  'unite',
  'some',
  'of',
  'the',
  'best',
  'blessings',
  'of',
  'existence',
  'and',
  'had',
  'lived',
  'nearly',
  'twenty',
  'one',
  'years',
  'in',
  'the',
  'world',
  'with',
  'very',
  'little',
  'to',
  'distress',
  'or',
  'vex',
  'her'],
 ['she',
  'was',
  'the',
  'youngest',
  'of',
  'the',
  'two',
  'daughters',
  'of',
  'a',
  'most',
  'affectionate',
  'indulgent',
  'father',
  'and',
  'had',
  'in',
  'consequence',
  'of',
  'her',
  'sister',
  's',
  'marriage',
  'been',
  'mistress',
  'of',
  'his',
  'house',
  'from',
  'a',
  'very',
  'early',
  'period'],
 ['her',
  'mother',
  'had',
  'died',
  'too',
  'long',
  'ago',
  'for',
  'her',
  'to',
  'have',
  'more',
  'than',
  'an',
 

In [28]:
clean_sents[6] # could consider removing stop words or common words

['sixteen',
 'years',
 'had',
 'miss_taylor',
 'been',
 'in',
 'mr_woodhouse',
 's',
 'family',
 'less',
 'as',
 'a',
 'governess',
 'than',
 'a',
 'friend',
 'very',
 'fond',
 'of',
 'both',
 'daughters',
 'but',
 'particularly',
 'of',
 'emma']

#### Run word2vec

In [29]:
# max_vocab_size can be used instead of min_count (which has increased here)
model = Word2Vec(sentences=clean_sents, size=64, sg=1, window=10, min_count=10, seed=42, workers=8)
model.save('clean_gutenberg_model.w2v')

#### Explore model

In [30]:
# skip re-training the model with the next line:  
model = gensim.models.Word2Vec.load('clean_gutenberg_model.w2v')

In [31]:
len(model.wv.vocab) # down from 17k in previous notebook

10329

In [32]:
model['ma_am']

/anaconda3/lib/python3.5/site-packages/ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  """Entry point for launching an IPython kernel.


array([-0.19394362, -0.09497917, -0.6130751 , -0.36811197, -0.08481188,
        0.11588141, -0.04132374,  0.41681704,  0.3140406 , -0.5824431 ,
        0.02051501, -0.81242883, -0.6018881 ,  0.5162385 , -0.09077752,
       -0.34719956,  0.2246821 , -0.0244391 , -0.46358362,  0.212208  ,
        0.776602  , -0.34278515,  0.01027616,  0.04741153, -0.19352536,
        0.12666382,  0.25301275,  0.09927005,  0.22926624, -0.07505203,
       -0.62834716,  0.16071288, -0.12412907, -0.13766558,  0.4125551 ,
       -0.41409555, -0.4300891 , -0.13577113,  0.18814853,  0.38669372,
       -0.2686344 , -0.16826516, -0.13112472,  0.25999975, -0.16560549,
        0.05369596,  0.03983536,  0.01109796,  0.2034622 ,  0.5991304 ,
       -0.07763731,  0.0531988 , -0.25697798, -0.11079809, -0.3728341 ,
       -0.2765924 ,  0.23972997, -0.1271025 ,  0.26000488,  0.8856857 ,
       -0.29481834, -0.155177  , -0.20574608, -0.4712187 ], dtype=float32)

In [33]:
model.most_similar('ma_am') 

/anaconda3/lib/python3.5/site-packages/ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  """Entry point for launching an IPython kernel.


[('betty', 0.8780678510665894),
 ('m_sure', 0.8608224987983704),
 ('madam', 0.8429377675056458),
 ('mamma', 0.8281928896903992),
 ('shouldn', 0.8251910209655762),
 ('rosamond', 0.8209366798400879),
 ('frederick', 0.8121393918991089),
 (";'", 0.8119815587997437),
 ('.--"', 0.8109865188598633),
 ('interrupting', 0.809975266456604)]

In [34]:
# swap woman and man
model.most_similar(positive=['ma_am', 'man'], negative=['woman']) 

/anaconda3/lib/python3.5/site-packages/ipykernel_launcher.py:2: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  


[('angrily', 0.7331258058547974),
 ('grimly', 0.715499758720398),
 ('impatiently', 0.7137595415115356),
 ('todhunter', 0.7136568427085876),
 ('hutton', 0.7079294323921204),
 ('hirsch', 0.7050608396530151),
 ("'--", 0.7005002498626709),
 ('frog', 0.6989611387252808),
 ('thoughtfully', 0.6986574530601501),
 ('bullet', 0.6965330839157104)]

In [35]:
model.most_similar(positive=['father', 'woman'], negative=['man']) 

/anaconda3/lib/python3.5/site-packages/ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).
  """Entry point for launching an IPython kernel.


[('mother', 0.776068389415741),
 ('sister', 0.7583142518997192),
 ('husband', 0.7512427568435669),
 ('wife', 0.7445505261421204),
 ('daughter', 0.7418741583824158),
 ('brother', 0.6830244064331055),
 ('daughters', 0.6829159259796143),
 ('dearly', 0.6668894290924072),
 ('loved', 0.6577752828598022),
 ('rachel', 0.6571471691131592)]

#### Reduce word vector dimensionality with t-SNE

In [36]:
tsne = TSNE(n_components=2, n_iter=1000)

In [37]:
X_2d = tsne.fit_transform(model[model.wv.vocab])

/anaconda3/lib/python3.5/site-packages/ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  """Entry point for launching an IPython kernel.


In [38]:
coords_df = pd.DataFrame(X_2d, columns=['x','y'])
coords_df['token'] = model.wv.vocab.keys()

In [39]:
coords_df.head()

,x,y,token
0,5.449492,44.703690,140
1,4.835161,-30.609018,rivet
2,-38.240501,-9.879745,bulk
3,-7.127518,-36.540619,cruet
4,-30.456543,7.877321,passages


In [40]:
# coords_df.to_csv('clean_gutenberg_tsne.csv', index=False)

#### Visualise 

In [41]:
coords_df = pd.read_csv('clean_gutenberg_tsne.csv')

FileNotFoundError: File b'clean_gutenberg_tsne.csv' does not exist

In [ ]:
_ = coords_df.plot.scatter('x', 'y', figsize=(12,12), marker='.', s=10, alpha=0.2)

In [ ]:
output_notebook()

In [ ]:
subset_df = coords_df.sample(n=5000)

In [ ]:
p = figure(plot_width=800, plot_height=800)
_ = p.text(x=subset_df.x, y=subset_df.y, text=subset_df.token)

In [ ]:
show(p)

In [ ]:
# output_file() here